# 03 · Comparative figures (across experiments)

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

Builds the **paper figures** (`configs/figures/paper.yaml`, `docs/figures.md`) that compare
experiments, **only from saved outputs**. Run it after the experiments you want to compare have
finished; it trains nothing and starts no experiment. Figures whose inputs do not exist yet are
skipped, and figures drawn with some inputs still missing are listed as INCOMPLETE (with what is
missing). Each run writes a new folder `figures/figures_<time>/` with figures, plotted data and a
manifest. Single-experiment diagnostics are in each experiment's own report folder instead.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 1. Which experiments have results

In [ ]:
from pathlib import Path
import pandas as pd
from adgraph.experiments.plan import experiments_overview
from adgraph.graphs.store import GraphStore
from adgraph.pipeline import latest_complete_graph_run

GRAPHS = latest_complete_graph_run(OUT)
RESULTS = f"{OUT}/experiments"
CONFIGS = ["configs/experiments/exp0_sensor_psd.yaml", "configs/experiments/exp1_source_psd.yaml", "configs/experiments/exp2_conn_minimal_fixedhp.yaml", "configs/experiments/exp3_conn_psd_fixedhp.yaml", "configs/experiments/exp4_band_mean_fixedhp.yaml", "configs/experiments/exp5_multiband_fixedhp.yaml", "configs/experiments/exp6_multiband_psd_strength_fixedhp.yaml", "configs/experiments/exp2_conn_minimal_fixedhp_tier2.yaml", "configs/experiments/exp3_conn_psd_fixedhp_tier2.yaml", "configs/experiments/exp5_multiband_fixedhp_tier2.yaml", "configs/experiments/exp6_multiband_psd_strength_fixedhp_tier2.yaml", "configs/experiments/exp3b_conn_psd_allbands_fixedhp.yaml", "configs/experiments/baseline_age.yaml", "configs/experiments/baseline_conn_classical.yaml", "configs/experiments/probe_dataset.yaml", "configs/experiments/age/exp0_sensor_psd_plus_age.yaml", "configs/experiments/age/exp1_source_psd_plus_age.yaml", "configs/experiments/age/exp2_conn_minimal_fixedhp_plus_age.yaml", "configs/experiments/age/exp3_conn_psd_fixedhp_plus_age.yaml", "configs/experiments/age/exp4_band_mean_fixedhp_plus_age.yaml", "configs/experiments/age/exp5_multiband_fixedhp_plus_age.yaml", "configs/experiments/age/exp6_multiband_psd_strength_fixedhp_plus_age.yaml", "configs/experiments/age/baseline_conn_classical_plus_age.yaml", "configs/experiments/exp7_earm_gnn.yaml"]
print('graph run:', GRAPHS)
pd.set_option('display.width', 250)
experiments_overview(CONFIGS, GraphStore.load(GRAPHS, keys=()), RESULTS)

## 2. Generate the comparative figures

In [ ]:
sh(f"python scripts/make_figures.py --config configs/figures/paper.yaml --results '{RESULTS}' "
   f"--graphs '{GRAPHS}' --out '{OUT}/figures'")
from IPython.display import Image, display
latest = sorted(Path(f"{OUT}/figures").glob('figures_*'))[-1]
print('figures in', latest)
for png in sorted(latest.glob('Fig*.png')):
    print(png.name)
    display(Image(filename=str(png)))
print('supplementary:', sorted(p.name for p in latest.glob('S*.png')))